In [ ]:
from fmsae_repro import DATA, MODELS, RESULTS, RAW_DATA
import ezslide
import lazyslide as zs
import mesoslide as ms
import pandas as pd

In [ ]:
slide = ezslide.read_slide(f'{DATA}/adjacent_registered/LSP24530.zarr', attach_images=False)
slide

In [ ]:
import joblib
sae = joblib.load(f'{MODELS}/ref_SAE.joblib')

In [ ]:
ms.tl.feature_extraction(
    slide,
    model="uni",
    tile_key="tiles",
    key_added="UNI_embedding",
    sparse=True,
    sparse_transform=sae.transform,
    sparse_key_added="UNI_SAE",
    save=True,
    batch_size=2048
)

In [ ]:
_ = ms.plotting.plot_feature_map(slide, 'UNI_SAE_7815', figsize=(6,6), return_fig=True)

In [ ]:
patches = ms.select_top_patches(slide, 'UNI_SAE_7815', 20, take_every=1)
_ = ms.plotting.plot_patch_gallery(patches, return_fig=True)

In [ ]:
la_patches = ms.select_top_patches(slide, 'UNI_SAE_27188', 20, top_fraction=0.1)
_ = ms.plotting.plot_patch_gallery(la_patches, return_fig=True)

In [ ]:
from matplotlib import pyplot as plt
plt.hist(patches_.obs['_feature_score'])

In [ ]:
cycif_path = f'{RAW_DATA}/3D/CyCIF/LSP24531.ome.tif'
cycif_slide = ezslide.open_slide(cycif_path, reader='tifffile_zarr', attach_images=False)
# cycif_slide

In [ ]:
zs.pp.find_tissues(cycif_slide, detect_holes=False)
zs.pl.tissue(cycif_slide)

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
plt.imshow(np.array(slide.reader[0][5][:]), alpha=0.5)
plt.imshow(cycif_slide.reader[0][5][0], alpha=0.5, vmax=20000)

In [ ]:
import os
import joblib
from mesoslide.preprocessing import extract_cluster_maps
import mesoslide as ms

OLD_CLUSTERIZERS_DIR = f"{MODELS}/token_clusterers/legacy"

SAE_FEATURE = "47380"

# 1. Load a converted clusterizer -- a real TokenClusterizer, model_name="uni" baked in.
clusterizer = joblib.load(os.path.join(OLD_CLUSTERIZERS_DIR, f"UNI_SAE_{SAE_FEATURE}.joblib"))

# 2. Select patches for that SAE feature.
patches = ms.select_top_patches(slide, f"UNI_SAE_{SAE_FEATURE}", n=20, top_fraction=0.1)
patches

In [ ]:
clusterizer.fit(slide, f'UNI_SAE_{SAE_FEATURE}')

In [ ]:
# 4. Visualize alongside H&E patches.
_ = ms.plotting.plot_patch_gallery_with_saliency(
    patches, clusterizers=[clusterizer],
    return_fig=True,
    cache=True,
    cmap='viridis'
)

In [ ]:
# 4. Visualize alongside H&E patches.
_ = ms.plotting.plot_patch_gallery_with_saliency(
    la_patches, clusterizers=[clusterizers_mapping[59985]],
    return_fig=True,
    cmap='viridis'
)

In [ ]:
import importlib
import mesoslide.preprocessing._extract_patches
importlib.reload(mesoslide.preprocessing._extract_patches)


In [ ]:
ms.preprocessing.extract_patch_images(patches, 
                     cycif_slide, 
                     channels=list(range(0, 49)), 
                     cache=True,
                     overwrite=True)

In [ ]:
import pandas as pd
from mesoslide.plotting import GalleryPlan

marker_table = pd.read_csv(f'{RAW_DATA}/3D/markers.csv')

plan = (
    GalleryPlan(patches)
    .add_he_row()  # slides=None -> falls back to patches.obs['_slide_ref'] (the H&E slide)
    .add_cluster_map_rows([clusterizer], cmap='viridis')  # already cached from cell 8
    .add_cycif_rows(
        ['DNA1', 'CD11c', 'CD206'],  # markers to show
        slides=cycif_slide,          # _slide_ref points at H&E, not CyCIF -- must pass explicitly
        marker_table=marker_table,
        merge=True,                  # adds a "Merge" row first, then CD11c, then CD206
        cache=True
    )
)
_ = plan.render(return_fig=True)


In [ ]:
seg_slide = ezslide.open_slide(f'{RAW_DATA}/3D/CyCIF/segmentation/LSP24531/cellRing.ome.tif', attach_images=False)
seg_slide

In [ ]:
import lazyslide as zs
v = zs.pl.WSIViewer(seg_slide)
v.add_image()                          # slide as background
# v.add_contours("tissues")              # optional overlays
# v.add_polygons("annotations")
# v.set_tissue_id(0)
v.add_zoom(0.6, 0.61, 0.3, 0.31)         # xmin, xmax, ymin, ymax
v.add_scalebar()
v.mark_origin()
v.show()     

In [ ]:
import numpy as np
from lazyslide.cv import InstanceMap
from wsidata.io import add_shapes

# h, w = seg_slide.properties.shape                      # (height, width) at level 0
# img = seg_slide.read_region(0, 0, w, h, level=0)       # (H, W, 3) uint8 in most readers

# label_img = (img[..., 0] if img.ndim == 3 else img).astype(np.int32)

# gdf = InstanceMap(label_img).to_polygons()           # background 0 is skipped automatically
# add_shapes(seg_slide, key="cells", shapes=gdf)

# rng = np.random.default_rng(0)
# palette = {i: (*rng.random(3), 1.0) for i in gdf["instance_id"]}

v = zs.pl.WSIViewer(seg_slide)
v.add_image()
v.add_polygons("cells", color_by="instance_id", palette=palette, alpha=0.7, linewidth=0, legend=False)
v.add_zoom(0.6, 0.61, 0.3, 0.31)
v.show()

The crash comes from the datashader backend, not your data. With more than 10,000 polygons, add_polygons switches to datashader, which makes one raster layer per unique instance_id. Your error shows the result: an array of shape (624, 744, 1355010), one layer per each of your ~1.35 million cells, which is 2.29 TiB. Random color per instance doesn't work at this scale, but you don't need it. Give each cell one of about 32 random colors instead. Neighbors will almost always differ, and the image looks the same.

Fix A: keep the whole-slide view with a zoom box

However, the following takes too long to run

In [ ]:
import numpy as np
import lazyslide as zs
from wsidata.io import add_shapes

K = 32
rng = np.random.default_rng(0)
gdf["color_group"] = rng.integers(0, K, len(gdf))
palette = {i: (*rng.random(3), 1.0) for i in range(K)}

add_shapes(seg_slide, key="cells_colored", shapes=gdf)   # new key, so it doesn't clash with "cells"

v = zs.pl.WSIViewer(seg_slide)          # a new viewer, since the old one cached "cells"
v.add_image()
v.add_polygons("cells_colored", color_by="color_group", palette=palette,
               alpha=0.7, linewidth=0, legend=False)
v.add_zoom(0.6, 0.61, 0.3, 0.31)
v.show()

Fix B: skip the whole slide and plot only your box

If you only care about the region, set the viewport to it. Only polygons intersecting that region are rendered. This is usually faster and sharper than a tiny zoom inset. Call it before add_polygons, because the polygon count is decided when the layer is added.

The following works and is fast

In [ ]:
h, w = seg_slide.properties.shape
x0, x1 = 0.6 * w, 0.61 * w
y0, y1 = 0.3 * h, 0.31 * h

v = zs.pl.WSIViewer(seg_slide)
v.set_viewport(x0, y0, x1 - x0, y1 - y0)   # (x, y, width, height) at level 0
v.add_image()
v.add_polygons("cells_colored", color_by="color_group", palette=palette,
               alpha=0.7, linewidth=0, legend=False)
v.add_scalebar()
v.show()

Another option: i want to color with K=1 (basically where are the cells) on the WSI and in the zoom with K=32

How it works, from the 0.12.0 source:

Call 1 has no color_by, so datashader draws every cell in one color, which is fast and needs almost no memory. in_zoom=False keeps this single-color layer out of the zoom box.
Call 2 creates two layers: a base-view raster and a zoom-only vector layer. Forcing backend="datashader" guarantees the zoom-only layer is created.
The list filter removes the base raster from call 2, so the whole-slide view stays single-color. It relies on the private _render_plans attribute, so it may break in other LazySlide versions. I wrote it against 0.12.0, and I haven't run it on your data.
The zoom box draws only the vector polygons that fall inside it. Keep the box small, as yours is.

In [ ]:
import numpy as np
import lazyslide as zs
from shapely.geometry import box
from wsidata.io import add_shapes

h, w = seg_slide.properties.shape
x0, x1 = 0.6 * w, 0.61 * w
y0, y1 = 0.3 * h, 0.31 * h

# 1) Pre-select the cells in the zoom box (spatial index, fast even for 1.35M cells)
idx = gdf.sindex.query(box(x0, y0, x1, y1), predicate="intersects")
add_shapes(seg_slide, key="cells_zoom", shapes=gdf.iloc[idx].reset_index(drop=True))

# 2) Whole slide: one color. Zoom box: K colors, using only the small table
v = zs.pl.WSIViewer(seg_slide)
v.add_image()
v.add_polygons("cells_colored", color="#D3F527", alpha=0.9,
               linewidth=0, in_zoom=False, legend=False)     # whole slide, single color
v.add_polygons("cells_zoom", color_by="color_group", palette=palette,
               alpha=0.7, linewidth=0, legend=False)          # only the zoom-box cells
v.add_zoom(x0, x1, y0, y1)
v.show()

### Use new mesoslide API to add cell polygons and phenotypes

In [ ]:
import ezslide
import mesoslide as ms
from mesoslide.plotting import GalleryPlan

# 1. Open the CyCIF/segmentation slide and ingest cells + phenotypes.
cycif_slide = ezslide.open_slide(
    f"{RAW_DATA}/3D/CyCIF/LSP24531.ome.tif",
    reader="tifffile_zarr",
    attach_images=False,
)

ms.tl.add_cell_polygons(cycif_slide, 
                        chunk_size=8192,
                        mask_path=f"{RAW_DATA}/3D/CyCIF/segmentation/LSP24531/cellRing.ome.tif")

ms.tl.add_cell_phenotypes(
    cycif_slide,
    phenotype_csv=f"{RAW_DATA}/3D/CyCIF/quantification/LSP24531.csv",
    phenotype_col="phenotype",   # the CSV already has this column
)

# sanity check
cycif_slide.shapes["cells"].head()
cycif_slide.tables["cells_phenotypes"].obs.head()

In [ ]:
import importlib
import mesoslide.plotting._image_grid as _image_grid
import mesoslide.plotting._cell_overlay as _cell_overlay
import mesoslide.plotting._gallery_plan as _gallery_plan
import mesoslide.preprocessing._extract_cluster_maps as _extract_cluster_maps
importlib.reload(_image_grid)     # the actual fix (cm.get_cmap -> matplotlib.colormaps[...])
importlib.reload(_cell_overlay)   # re-binds its `from ._image_grid import _group_color_lookup`
importlib.reload(_gallery_plan)   # re-binds its `from ._cell_overlay import ...`
importlib.reload(_extract_cluster_maps)  # re-binds its `from ._gallery_plan import GalleryPlan`
# Re-fetch GalleryPlan directly from the reloaded module -- `mesoslide.plotting.GalleryPlan`
# (via ms.pl.GalleryPlan or `from mesoslide.plotting import GalleryPlan`) is lazy_loader-cached
# and won't pick up the reload on its own.
GalleryPlan = _gallery_plan.GalleryPlan

# also refresh the package-level caches so `ms.pl.GalleryPlan` / `mesoslide.plotting.GalleryPlan`
# stay consistent if you use those spellings elsewhere in the notebook
import mesoslide.plotting as _plotting
_plotting.GalleryPlan = GalleryPlan


In [ ]:
import pandas as pd
from mesoslide.plotting import GalleryPlan

marker_table = pd.read_csv(f'{RAW_DATA}/3D/markers.csv')

plan = (
    GalleryPlan(patches)
    .add_he_row(slides=slide)
    .add_cluster_map_rows([clusterizer], cmap='viridis')  # already cached from cell 8
    .add_cell_overlay_row(
        cells_key="cells",
        slides=cycif_slide,   # keyed by slide_id, since cells live on a separate slide
        color_by="phenotype",
        fill_alpha=0.4,
        edge_color="white",
        blend_with_previous=False
    )
    .add_cycif_rows(
        ['DNA1', 'CD11c', 'CD206'],  # markers to show
        slides=cycif_slide,          # _slide_ref points at H&E, not CyCIF -- must pass explicitly
        marker_table=marker_table,
        merge=True,                  # adds a "Merge" row first, then CD11c, then CD206
    )
)

_ = plan.render(return_fig=True)

### Foreground vs. background CyCIF marker intensity

`UNI_SAE_47380_cluster_img` rasterizes each patch's token-cluster assignment to pixel
resolution: label 2 is the SAE-feature foreground region, labels 0 and 1 are both
background. `cycif_patch_img` holds all 49 CyCIF channels at the same per-patch pixel
resolution (both were rasterized/read against the same H&E tile size, since the CyCIF
slide itself was never tiled -- `extract_patch_images`/`extract_cluster_maps` fall back
to the reference (H&E) slide's own tile spec in that case). For each of the 20 patches
and each of the 49 channels, compute the mean pixel intensity inside the foreground mask
and inside the background mask, then compare the two distributions (grouped box plots)
and their average log2 fold enrichment (waterfall plot).

In [ ]:
import numpy as np
import pandas as pd

CLUSTER_KEY = f"UNI_SAE_{SAE_FEATURE}_cluster_img"
FOREGROUND_LABEL = 2

cluster_img = patches.tables["tiles_table"].obsm[CLUSTER_KEY]   # (N, H, W) cluster ids
cycif_img = patches.tables["tiles_table"].obsm["cycif_patch_img"]  # (N, C, H, W) intensities

print("cluster_img:", cluster_img.shape, cluster_img.dtype)
print("cycif_img:  ", cycif_img.shape, cycif_img.dtype)

n_patches, n_channels = cycif_img.shape[0], cycif_img.shape[1]
assert cluster_img.shape[0] == n_patches, "cluster_img/cycif_img patch counts must match"

# Both are normally rasterized against the same (H&E) reference tile size, but resize
# defensively (nearest-neighbor, to preserve integer cluster labels) if they ever differ.
if cluster_img.shape[-2:] != cycif_img.shape[-2:]:
    import cv2
    target_h, target_w = cycif_img.shape[-2:]
    cluster_img = np.stack([
        cv2.resize(cluster_img[i].astype(np.int32), (target_w, target_h), interpolation=cv2.INTER_NEAREST)
        for i in range(n_patches)
    ])
    print("Resized cluster_img to match cycif_img:", cluster_img.shape)

fg_mask = cluster_img == FOREGROUND_LABEL   # (N, H, W)
bg_mask = ~fg_mask                          # labels 0 and 1

# Channel names, in the same 0-based order used by extract_patch_images(channels=range(49)).
marker_names = marker_table["marker_name"].tolist()
assert len(marker_names) == n_channels, (
    f"marker_table has {len(marker_names)} rows but cycif_img has {n_channels} channels"
)

fg_mean = np.full((n_patches, n_channels), np.nan)
bg_mean = np.full((n_patches, n_channels), np.nan)
for i in range(n_patches):
    fg_px, bg_px = fg_mask[i], bg_mask[i]
    for c in range(n_channels):
        chan = cycif_img[i, c].astype(np.float64)
        if fg_px.any():
            fg_mean[i, c] = chan[fg_px].mean()
        if bg_px.any():
            bg_mean[i, c] = chan[bg_px].mean()

fg_frac = fg_mask.reshape(n_patches, -1).mean(axis=1)
print(f"Foreground pixel fraction per patch: min={fg_frac.min():.3f}, max={fg_frac.max():.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Order channels by log2 fold enrichment (computed below) so the box plot and the
# waterfall plot read left-to-right/top-to-bottom in the same order.
log2fc = np.log2(np.nanmean(fg_mean, axis=0) / np.nanmean(bg_mean, axis=0))
order = np.argsort(-log2fc)
ordered_names = [marker_names[c] for c in order]

long_df = pd.concat([
    pd.DataFrame({
        "channel": np.repeat(marker_names, n_patches),
        "intensity": fg_mean.T.reshape(-1),
        "region": "foreground",
    }),
    pd.DataFrame({
        "channel": np.repeat(marker_names, n_patches),
        "intensity": bg_mean.T.reshape(-1),
        "region": "background",
    }),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(max(14, 0.4 * n_channels), 6))
sns.boxplot(
    data=long_df, x="channel", y="intensity", hue="region",
    order=ordered_names, hue_order=["foreground", "background"],
    palette={"foreground": "#d62728", "background": "#7f7f7f"},
    fliersize=2, linewidth=0.8, ax=ax,
)
ax.set_yscale("log")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_xlabel("")
ax.set_ylabel("Mean intensity per patch (log scale)")
ax.set_title(f"Foreground (cluster {FOREGROUND_LABEL}) vs. background CyCIF intensity, per channel (n={n_patches} patches)")
ax.legend(title=None, loc="upper right")
fig.tight_layout()

In [ ]:
fc_df = pd.DataFrame({"channel": marker_names, "log2fc": log2fc}).sort_values(
    "log2fc", ascending=False
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, max(6, 0.28 * n_channels)))
colors = ["#d62728" if v > 0 else "#1f77b4" for v in fc_df["log2fc"]]
ax.barh(fc_df["channel"], fc_df["log2fc"], color=colors)
ax.invert_yaxis()  # highest enrichment at the top
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel(f"log2(mean foreground / mean background)  [cluster {FOREGROUND_LABEL} vs. rest]")
ax.set_title(f"Average CyCIF marker enrichment in the SAE feature foreground (n={n_patches} patches)")
fig.tight_layout()

# fc_df

### Foreground/background enrichment for a myeloid phenotype set of interest

Cell type set of interest: `CD11c+ DC/Mac`, `CD206+ Mac`, `CD11c+ CD206+ Mac`, vs. every
other phenotype in `patches.shapes['cells']`.

1. Per patch, correlate the number of foreground (cluster 2) tokens against the number of
   cells of interest in that patch (Spearman, since we only care about monotonic
   association, not a specific functional form).
2. Pool cells across all 20 patches and compute the empirical
   `P(cell in set | cell in foreground)` vs. `P(cell in set | cell in background)`, by
   looking up each cell's centroid in the same rasterized `cluster_img` used above.

In [ ]:
INTEREST_PHENOTYPES = {"CD11c+ DC/Mac", "CD206+ Mac", "CD11c+ CD206+ Mac"}

cells_gdf = patches.shapes["cells"]
patch_df = patches.obs
x_arr = patch_df["x"].to_numpy()
y_arr = patch_df["y"].to_numpy()

# Foreground-pixel count per patch, as a proxy for foreground-token count: cluster_img
# is a uniform per-token upsample to pixel resolution, so pixel count is proportional
# to token count by the same constant factor in every patch (correlation is unaffected).
n_fg_tokens = (fg_mask.reshape(n_patches, -1).sum(axis=1) / 725).astype(int)  # 725 = 29x25, the patch size in tokens

n_cells_of_interest = np.zeros(n_patches, dtype=int)
n_cells_total = np.zeros(n_patches, dtype=int)
for i in range(n_patches):
    sub = cells_gdf[cells_gdf["patch_idx"] == i]
    n_cells_total[i] = len(sub)
    n_cells_of_interest[i] = sub["phenotype"].isin(INTEREST_PHENOTYPES).sum()

corr_df = pd.DataFrame({
    "patch_idx": np.arange(n_patches),
    "n_fg_tokens": n_fg_tokens,
    "n_cells_total": n_cells_total,
    "n_cells_of_interest": n_cells_of_interest,
})
corr_df

In [ ]:
from scipy.stats import spearmanr

rho, pval = spearmanr(corr_df["n_fg_tokens"], corr_df["n_cells_of_interest"])

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(corr_df["n_fg_tokens"], corr_df["n_cells_of_interest"], s=70, edgecolor="black", color="#d62728")
# for _, row in corr_df.iterrows():
#     ax.annotate(str(int(row["patch_idx"])), (row["n_fg_tokens"], row["n_cells_of_interest"]),
#                 textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xlabel("# foreground tokens per patch")
ax.set_ylabel("# cells of interest per patch\n(CD11c+ DC/Mac, CD206+ Mac, CD11c+ CD206+ Mac)")
ax.set_title(f"Spearman r={rho:.2f}, p={pval:.3g}  (n={n_patches} patches)")
fig.tight_layout()

In [ ]:
# Assign every cell to foreground/background by looking up its centroid, shifted from
# WSI/level-0 pixel coordinates to patch-local pixel coordinates, in the same rasterized
# cluster_img used above (both are 1:1 level-0 pixel resolution -- no downsampling).
h_px, w_px = fg_mask.shape[1:]

fg_count_of_interest = 0
fg_count_total = 0
bg_count_of_interest = 0
bg_count_total = 0

for i in range(n_patches):
    sub = cells_gdf[cells_gdf["patch_idx"] == i]
    if len(sub) == 0:
        continue
    centroids = sub.geometry.centroid
    local_x = np.clip((centroids.x.to_numpy() - x_arr[i]).round().astype(int), 0, w_px - 1)
    local_y = np.clip((centroids.y.to_numpy() - y_arr[i]).round().astype(int), 0, h_px - 1)
    is_fg = fg_mask[i][local_y, local_x]
    is_interest = sub["phenotype"].isin(INTEREST_PHENOTYPES).to_numpy()

    fg_count_of_interest += int((is_fg & is_interest).sum())
    fg_count_total += int(is_fg.sum())
    bg_count_of_interest += int((~is_fg & is_interest).sum())
    bg_count_total += int((~is_fg).sum())

p_interest_given_fg = fg_count_of_interest / fg_count_total
p_interest_given_bg = bg_count_of_interest / bg_count_total

print(f"P(cell in set | foreground) = {p_interest_given_fg:.4f}  ({fg_count_of_interest}/{fg_count_total} cells)")
print(f"P(cell in set | background) = {p_interest_given_bg:.4f}  ({bg_count_of_interest}/{bg_count_total} cells)")
print(f"Enrichment ratio (fg/bg): {p_interest_given_fg / p_interest_given_bg:.2f}x")

fig, ax = plt.subplots(figsize=(4, 5))
ax.bar(["foreground", "background"], [p_interest_given_fg, p_interest_given_bg],
       color=["#d62728", "#7f7f7f"])
ax.set_ylabel("P(cell in phenotype set of interest | region)")
ax.set_title(f"Myeloid phenotype enrichment by SAE cluster {FOREGROUND_LABEL} region\n(pooled across {n_patches} patches)")
fig.tight_layout()

Now we extract patches dataset for each of the four features

In [ ]:
import joblib
import os
from tqdm import tqdm

selected_features = [11854, 7815, 27188, 47380] # 59985 replaced
# clusterizers_mapping = {}
# patches_mapping = {}

for feature in tqdm(selected_features):
    if feature in clusterizers_mapping:
        print(f"Skipping feature {feature} -- already processed.")
        continue
    _clusterizer = joblib.load(os.path.join(OLD_CLUSTERIZERS_DIR, f"UNI_SAE_{feature}.joblib"))
    clusterizers_mapping[feature] = _clusterizer
    _clusterizer.fit(slide, f'UNI_SAE_{feature}')
    joblib.dump(_clusterizer, os.path.join(OLD_CLUSTERIZERS_DIR, f"UNI_SAE_{feature}.joblib"))
    
    _patches = ms.select_top_patches(slide, f"UNI_SAE_{feature}", n=20, top_fraction=0.1)
    patches_mapping[feature] = _patches
    
    _ = ms.plotting.plot_patch_gallery_with_saliency(
        _patches, clusterizers=[_clusterizer],
        return_fig=True,
        cache=True,
        cmap='viridis'
    )


In [ ]:
for feature, _patches in patches_mapping.items():
    ms.preprocessing.extract_patch_images(_patches, 
                         cycif_slide, 
                         channels=list(range(0, 49)), 
                         cache=True,
                         overwrite=True)

In [ ]:
import os
import pandas as pd
from mesoslide.plotting import GalleryPlan

OUTPUT_DIR = f"{RESULTS}/figures/multimodal_panels"
os.makedirs(OUTPUT_DIR, exist_ok=True)

marker_table = pd.read_csv(f'{RAW_DATA}/3D/markers.csv')
markers_mapping = {
    11854: ['DNA1', 'aSMA', 'CD31', 'panCK'],
    7815: ['DNA1', 'panCK', 'CD11c', 'aSMA'],
    27188: ['DNA1', 'CD20', 'CD21', 'CD3E'],
    47380: ['DNA1', 'CD206', 'panCK', 'CD3E'],
    # 5840: ['DNA1', 'panCK', 'CD3E', 'CD11c']
}

# Phenotype set of interest per feature -- must match INTEREST_PHENOTYPES_BY_FEATURE
# defined later in the notebook (duplicated here so this cell is self-contained and can
# run before that cell).
INTEREST_PHENOTYPES_BY_FEATURE = {
    11854: {"Endothelial cell"},
    7815:  {"Epithelial cell"},
    27188: {"B cell", "CD4 T cell", "CD8 T cell", "Other T cell", "Treg"},
    47380: {"CD11c+ DC/Mac", "CD206+ Mac", "CD11c+ CD206+ Mac"},
}
all_phenotypes = sorted(cycif_slide.tables["cells_phenotypes"].obs["phenotype"].unique())

# render() only accepts a single in-memory figure (return_fig=True) when everything
# fits on one page; once patches_per_row/samples_per_figure force multiple pages, it
# requires output_path (and/or return_buffer=True) and saves each page itself, inserting
# the patch range before the extension via `_paged_path` (e.g. "feature_11854_1-5.png").
PATCHES_PER_ROW = 10
PATCHES_PER_PAGE = 10  # -> ceil(20 / 10) = 2 pages per feature

for feature, _patches in patches_mapping.items():
    if not feature == 47380:
        continue
    selected_markers = markers_mapping.get(feature, [])
    interest_phenotypes = INTEREST_PHENOTYPES_BY_FEATURE.get(feature, set())
    phenotype_palette = {
        p: "yellow" if p in interest_phenotypes else "purple" for p in all_phenotypes
    }
    plan = (
        GalleryPlan(_patches)
        .add_he_row()  # slides=None -> falls back to patches.obs['_slide_ref'] (the H&E slide)
        .add_cluster_map_rows([clusterizers_mapping[feature]], 
                              cmap='viridis')  # already cached from cell 8
        .add_cell_overlay_row(
            cells_key="cells",
            slides=cycif_slide,   # keyed by slide_id, since cells live on a separate slide
            color_by="phenotype",
            palette=phenotype_palette,  # yellow = phenotype set of interest, purple = everything else
            fill_alpha=0.4,
            edge_color="white",
            blend_with_previous=False
        )
        .add_cycif_rows(
            selected_markers,  # markers to show
            marker_table=marker_table,
            merge=True,                  # adds a "Merge" row first, then CD11c, then CD206
            vmin={'CD3E': 500},
            vmax={'CD3E': 5000}
        )
    )
    plan.render(
        output_path=os.path.join(OUTPUT_DIR, f"feature_{feature}.png"),
        patches_per_row=PATCHES_PER_ROW,
        samples_per_figure=PATCHES_PER_PAGE,
    )

In [ ]:
plt.imshow([[0,1,2]], cmap='viridis')

### Foreground vs. background CyCIF marker intensity across all four SAE features

Repeat the foreground/background analysis above for each of the four selected SAE
features (`selected_features`), using the per-feature `patches_mapping[feature]` /
`clusterizers_mapping[feature]` extracted earlier. For the box plots, each channel's
foreground and background intensities are scaled independently so that channel's
background mean equals 1 arbitrary unit (a.u.), making magnitudes comparable across
channels within a plot. The heatmap summarizes all four features' log2 fold enrichment
(foreground / background) side by side, with each row's 49 markers independently sorted
by enrichment.

In [ ]:
import numpy as np
import pandas as pd

markers_to_exclude = ['DNA2', 'DNA3', 'DNA4', 'DNA5', 'DNA6', 'DNA7', 'DNA8', 'DNA9', 'DNA10', 'DNA11',
                      'Ag85B', 'CD86', 'PD1', 'PDL1', 'CD163', 'CD11b', 'IRF-4',
                      'CD45', 'TCF1_TCF7', 'ICOS', 'HLADRB1', 'CD68', 'CTLA4', 'LaminB1AC', 'NGFR', 'ABCG1']
FOREGROUND_LABEL = 2
FOREGROUND_LABELS_BY_FEATURE = {27188: [1, 2]}  # feature 59985 uses labels 1 and 2 as foreground; all others use FOREGROUND_LABEL

feature_stats = {}  # feature -> {fg_mean, bg_mean, log2fc, marker_names, n_patches, foreground_labels}

for feature in selected_features:
    _patches = patches_mapping[feature]
    cluster_key = f"UNI_SAE_{feature}_cluster_img"
    _foreground_labels = FOREGROUND_LABELS_BY_FEATURE.get(feature, [FOREGROUND_LABEL])

    _cluster_img = _patches.tables["tiles_table"].obsm[cluster_key]      # (N, H, W)
    _cycif_img = _patches.tables["tiles_table"].obsm["cycif_patch_img"]  # (N, C, H, W)

    _n_patches, _n_channels = _cycif_img.shape[0], _cycif_img.shape[1]
    assert _cluster_img.shape[0] == _n_patches, "cluster_img/cycif_img patch counts must match"

    if _cluster_img.shape[-2:] != _cycif_img.shape[-2:]:
        import cv2
        target_h, target_w = _cycif_img.shape[-2:]
        _cluster_img = np.stack([
            cv2.resize(_cluster_img[i].astype(np.int32), (target_w, target_h), interpolation=cv2.INTER_NEAREST)
            for i in range(_n_patches)
        ])

    _fg_mask = np.isin(_cluster_img, _foreground_labels)
    _bg_mask = ~_fg_mask

    _marker_names = marker_table["marker_name"].tolist()
    assert len(_marker_names) == _n_channels, (
        f"marker_table has {len(_marker_names)} rows but cycif_img has {_n_channels} channels"
    )

    _fg_mean = np.full((_n_patches, _n_channels), np.nan)
    _bg_mean = np.full((_n_patches, _n_channels), np.nan)
    for i in range(_n_patches):
        fg_px, bg_px = _fg_mask[i], _bg_mask[i]
        for c in range(_n_channels):
            chan = _cycif_img[i, c].astype(np.float64)
            if fg_px.any():
                _fg_mean[i, c] = chan[fg_px].mean()
            if bg_px.any():
                _bg_mean[i, c] = chan[bg_px].mean()

    _log2fc = np.log2(np.nanmean(_fg_mean, axis=0) / np.nanmean(_bg_mean, axis=0))

    # Drop excluded markers before storing, so all downstream plots skip them.
    _keep = [c for c, name in enumerate(_marker_names) if name not in markers_to_exclude]
    _marker_names = [_marker_names[c] for c in _keep]
    _fg_mean = _fg_mean[:, _keep]
    _bg_mean = _bg_mean[:, _keep]
    _log2fc = _log2fc[_keep]

    feature_stats[feature] = {
        "fg_mean": _fg_mean,
        "bg_mean": _bg_mean,
        "log2fc": _log2fc,
        "marker_names": _marker_names,
        "n_patches": _n_patches,
        "foreground_labels": _foreground_labels,
    }

print({f: (s["n_patches"], s["foreground_labels"], len(s["marker_names"])) for f, s in feature_stats.items()})

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for feature in selected_features:
    stats = feature_stats[feature]
    fg_mean, bg_mean = stats["fg_mean"], stats["bg_mean"]
    marker_names, log2fc, n_patches = stats["marker_names"], stats["log2fc"], stats["n_patches"]
    foreground_labels = stats["foreground_labels"]
    n_channels = len(marker_names)

    # Scale each channel independently so its background mean equals 1 a.u.
    bg_scale = np.nanmean(bg_mean, axis=0)
    fg_norm = fg_mean / bg_scale
    bg_norm = bg_mean / bg_scale

    order = np.argsort(-log2fc)
    ordered_names = [marker_names[c] for c in order]

    long_df = pd.concat([
        pd.DataFrame({
            "channel": np.repeat(marker_names, n_patches),
            "intensity": fg_norm.T.reshape(-1),
            "region": "Salient tokens",
        }),
        pd.DataFrame({
            "channel": np.repeat(marker_names, n_patches),
            "intensity": bg_norm.T.reshape(-1),
            "region": "Background",
        }),
    ], ignore_index=True)

    fig, ax = plt.subplots(figsize=(max(14, 0.4 * n_channels), 6))
    sns.boxplot(
        data=long_df, x="channel", y="intensity", hue="region",
        order=ordered_names, hue_order=["Salient tokens", "Background"],
        palette={"Salient tokens": "#d62728", "Background": "#7f7f7f"},
        fliersize=2, linewidth=0.8, ax=ax,
    )
    ax.axhline(1, color="black", linewidth=0.8, linestyle="--")
    ax.set_yscale("log")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_xlabel("")
    ax.set_ylabel("MFI (a.u.)")
    # ax.set_title(
    #     f"SAE feature {feature}: foreground (cluster {foreground_labels}) vs. background "
    #     f"CyCIF intensity, per channel (n={n_patches} patches, {n_channels} markers shown)"
    # )
    ax.set_title(
        f"SAE feature {feature}"
    )
    ax.legend(title=None, loc="upper right")
    fig.tight_layout()

In [ ]:
column_names = feature_stats[selected_features[0]]["marker_names"]  # marker_table order, post-exclusion
n_markers = len(column_names)

heat_arr = np.full((len(selected_features), n_markers), np.nan)

for r, feature in enumerate(selected_features):
    stats = feature_stats[feature]
    _lookup = dict(zip(stats["marker_names"], stats["log2fc"]))
    heat_arr[r] = [_lookup[name] for name in column_names]

vmax = np.nanmax(np.abs(heat_arr))

fig, ax = plt.subplots(figsize=(0.35 * n_markers, 0.6 * len(selected_features) + 1))
im = ax.imshow(heat_arr, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")

ax.set_yticks(range(len(selected_features)))
ax.set_yticklabels([f"{feature}" for feature in selected_features])
ax.set_xticks(range(n_markers))
ax.set_xticklabels(column_names, rotation=90, fontsize=6)
# ax.set_xlabel("Markers")
ax.set_ylabel("SAE feature")
ax.set_title("CyCIF marker enrichment")
fig.colorbar(im, ax=ax, label="log2FC (Salient tokens / Background)", fraction=0.02, pad=0.01)
fig.tight_layout()

### Foreground/background enrichment for phenotype sets of interest, across SAE features

Generalizes the myeloid-phenotype analysis above (originally specific to feature 47380)
to all four selected SAE features. For each feature, given a chosen set of cell-type
phenotypes of interest:

1. Per patch, correlate the number of foreground tokens against the number of cells of
   interest in that patch (Spearman).
2. Pool cells across all patches for that feature and compute the empirical
   `P(cell in set | cell in foreground)` vs. `P(cell in set | cell in background)`, by
   looking up each cell's centroid in the same rasterized foreground mask used for the
   intensity analysis above.

The phenotype sets below are provisional, picked to roughly match each feature's marker
panel (see `markers_mapping`); the cell below first prints the actual phenotype
categories present in the data so you can verify/adjust
`INTEREST_PHENOTYPES_BY_FEATURE`.

In [ ]:
all_phenotypes = sorted(cycif_slide.tables["cells_phenotypes"].obs["phenotype"].unique())
print(f"{len(all_phenotypes)} phenotype categories in the data:")
for p in all_phenotypes:
    print(" ", p)

In [ ]:
from scipy.stats import spearmanr

# Provisional cell-type sets of interest per feature, matched loosely to each feature's
# marker panel in `markers_mapping`. Verify against `all_phenotypes` above and adjust.
INTEREST_PHENOTYPES_BY_FEATURE = {
    11854: {"Endothelial cell"},           
    7815:  {"Epithelial cell"},                                     
    27188: {"B cell", "CD4 T cell", "CD8 T cell", "Other T cell", "Treg"},         
    47380: {"CD11c+ DC/Mac", "CD206+ Mac", "CD11c+ CD206+ Mac"}
}
TOKENS_PER_PATCH = 725 

phenotype_enrichment = {}  # feature -> {corr_df, rho, pval, p_interest_given_fg/bg, interest_phenotypes}

for feature in selected_features:
    _patches = patches_mapping[feature]
    stats = feature_stats[feature]
    interest_phenotypes = INTEREST_PHENOTYPES_BY_FEATURE.get(feature, set())
    missing = interest_phenotypes - set(all_phenotypes)
    if missing:
        print(f"Feature {feature}: WARNING -- phenotypes not found in data: {missing}")

    # Rebuild the rasterized foreground mask for this feature (same recipe used above).
    cluster_key = f"UNI_SAE_{feature}_cluster_img"
    _cluster_img = _patches.tables["tiles_table"].obsm[cluster_key]
    _cycif_img = _patches.tables["tiles_table"].obsm["cycif_patch_img"]
    if _cluster_img.shape[-2:] != _cycif_img.shape[-2:]:
        import cv2
        target_h, target_w = _cycif_img.shape[-2:]
        _cluster_img = np.stack([
            cv2.resize(_cluster_img[i].astype(np.int32), (target_w, target_h), interpolation=cv2.INTER_NEAREST)
            for i in range(_cluster_img.shape[0])
        ])
    _fg_mask = np.isin(_cluster_img, stats["foreground_labels"])
    _n_patches = _fg_mask.shape[0]

    cells_gdf = _patches.shapes["cells"]
    patch_df = _patches.obs
    x_arr = patch_df["x"].to_numpy()
    y_arr = patch_df["y"].to_numpy()

    # 1. Per-patch correlation: # foreground tokens vs. # cells of interest.
    n_fg_tokens = (_fg_mask.reshape(_n_patches, -1).sum(axis=1) / TOKENS_PER_PATCH).astype(int)
    n_cells_of_interest = np.zeros(_n_patches, dtype=int)
    n_cells_total = np.zeros(_n_patches, dtype=int)
    for i in range(_n_patches):
        sub = cells_gdf[cells_gdf["patch_idx"] == i]
        n_cells_total[i] = len(sub)
        n_cells_of_interest[i] = sub["phenotype"].isin(interest_phenotypes).sum()

    corr_df = pd.DataFrame({
        "patch_idx": np.arange(_n_patches),
        "n_fg_tokens": n_fg_tokens,
        "n_cells_total": n_cells_total,
        "n_cells_of_interest": n_cells_of_interest,
    })
    rho, pval = spearmanr(corr_df["n_fg_tokens"], corr_df["n_cells_of_interest"])

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(
        corr_df["n_fg_tokens"], corr_df["n_cells_of_interest"], s=70, edgecolor="black", color="#d62728",
        label=f"Spearman r={rho:.2f}, p={pval:.3g}",
    )
    ax.set_xlabel("# salient tokens")
    ax.set_ylabel(f"# cells in phenotype set")
    ax.set_title(f"SAE feature {feature}")
    ax.legend(loc="best")
    fig.tight_layout()

    # 2. Pooled empirical P(cell in set | foreground) vs. P(cell in set | background).
    h_px, w_px = _fg_mask.shape[1:]
    fg_count_of_interest = fg_count_total = bg_count_of_interest = bg_count_total = 0
    for i in range(_n_patches):
        sub = cells_gdf[cells_gdf["patch_idx"] == i]
        if len(sub) == 0:
            continue
        centroids = sub.geometry.centroid
        local_x = np.clip((centroids.x.to_numpy() - x_arr[i]).round().astype(int), 0, w_px - 1)
        local_y = np.clip((centroids.y.to_numpy() - y_arr[i]).round().astype(int), 0, h_px - 1)
        is_fg = _fg_mask[i][local_y, local_x]
        is_interest = sub["phenotype"].isin(interest_phenotypes).to_numpy()

        fg_count_of_interest += int((is_fg & is_interest).sum())
        fg_count_total += int(is_fg.sum())
        bg_count_of_interest += int((~is_fg & is_interest).sum())
        bg_count_total += int((~is_fg).sum())

    p_interest_given_fg = fg_count_of_interest / fg_count_total if fg_count_total else np.nan
    p_interest_given_bg = bg_count_of_interest / bg_count_total if bg_count_total else np.nan

    fig, ax = plt.subplots(figsize=(4, 5))
    ax.bar(["Salient tokens", "Background"], [p_interest_given_fg, p_interest_given_bg],
           color=["#d62728", "#7f7f7f"])
    ax.set_ylabel("Conditional probability of cell in phenotype set")
    ax.set_title(
        f"SAE feature {feature}"
    )
    fig.tight_layout()

    phenotype_enrichment[feature] = {
        "corr_df": corr_df,
        "rho": rho,
        "pval": pval,
        "p_interest_given_fg": p_interest_given_fg,
        "p_interest_given_bg": p_interest_given_bg,
        "interest_phenotypes": interest_phenotypes,
    }
    print(f"Feature {feature}: rho={rho:.3f} p={pval:.3g}  "
          f"P(interest|fg)={p_interest_given_fg:.4f}  P(interest|bg)={p_interest_given_bg:.4f}")